In [ ]:
# -*- coding: utf-8 -*-

import os
import torch
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
# Consider using Pillow-SIMD for potentially faster image ops, install with:
# pip uninstall pillow && pip install Pillow-SIMD
# No code change needed if installed, it replaces Pillow transparently.
from PIL import Image, ImageFile
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import timm
import logging
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import glob
from typing import Optional, Tuple, List, Dict
import torch.nn.functional as F # Needed for pooling, activations etc.
from sklearn.utils import shuffle
from torch.cuda.amp import GradScaler, autocast # Automatic Mixed Precision
import time # For timing epochs
import platform # For PyTorch version check

# Allow loading of truncated images (use with caution)
ImageFile.LOAD_TRUNCATED_IMAGES = True

# --- Configuration ---
class Config:
    # Data Params
    # --- IMPORTANT: VERIFY AND SET THESE PATHS ---
    # Suggest a NEW directory for preprocessed data specific to this model/config
    PREPROCESSED_DIR = "/content/drive/MyDrive/preprocessed_casia_earlyfusion_optimized_v2" # Use a distinct name
    # Path to the ROOT directory containing the CASIA dataset (e.g., contains 'Au' and 'Tp' subdirs)
    DATASET_ROOT = "/content/drive/MyDrive/CASIA2.0_revised_corrected/casia"
    # --- END IMPORTANT PATHS ---
    DATASET_FRACTION = 1.0 # Use 1.0 for full dataset, smaller for testing
    SEED = 42
    IMG_SIZE = 224
    NUM_CLASSES = 2  # CASIA has 2 classes: authentic and tampered
    MEAN = [0.485, 0.456, 0.406]  # ImageNet mean
    STD = [0.229, 0.224, 0.225]  # ImageNet std

    # Training Params
    BATCH_SIZE = 64 # <<< OPTIMIZATION: INCREASE BATCH SIZE if GPU memory allows
    NUM_WORKERS = 4 # <<< OPTIMIZATION: Set based on CPU cores and system (try 2, 4, 8)
    EPOCHS_PHASE1 = 5
    EPOCHS_PHASE2 = 10
    LEARNING_RATE_PHASE1 = 1e-4 # Keep separate LRs for phases
    LEARNING_RATE_PHASE2 = 1e-5 # Lower LR for fine-tuning
    EARLY_STOPPING_PATIENCE = 5
    CLIP_GRAD_NORM = 1.0 # Gradient clipping helps stabilize training
    OPTIMIZER = 'AdamW' # Options: 'AdamW', 'SGD' etc. AdamW often good default.
    WEIGHT_DECAY = 0.01 # Weight decay for AdamW

    # Model Params
    MODEL_NAME = "early_fusion_swin_effnet_optimized_v2" # Descriptive name for saved files
    # Use torch.compile if available (PyTorch 2.0+) for potential speedup
    USE_TORCH_COMPILE = True # <<< OPTIMIZATION: Set to True if using PyTorch 2.0+

    # Early Fusion Specific Params
    FUSION_TARGET_SPATIAL_SIZE = 7
    FUSION_CONV_CHANNELS = 512
    FUSION_DROPOUT = 0.3

    # Derived/Runtime Params (Don't manually set these)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
    PIN_MEMORY = (DEVICE.type == 'cuda') # Pin memory only if using CUDA
    # Check PyTorch version for torch.compile
    TORCH_MAJOR_VERSION = int(platform.python_version_tuple()[0])
    COMPILE_MODEL = USE_TORCH_COMPILE and TORCH_MAJOR_VERSION >= 2 and DEVICE.type == 'cuda' # Compile only for CUDA on PyTorch 2+


config = Config()

# --- Setup logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
log_file = f"{config.MODEL_NAME}_training.log"
file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))
logging.getLogger().addHandler(file_handler)


# --- Set random seeds for reproducibility ---
def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if config.DEVICE.type == 'cuda':
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # if using multi-GPU
        # <<< OPTIMIZATION: Benchmark mode can be faster if input sizes don't change
        # Set deterministic only if exact reproducibility is absolutely required, it can slow down training.
        # torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        logging.info("cuDNN Benchmark enabled.")
    # else:
    #     torch.backends.cudnn.deterministic = False # Ensure it's off if not CUDA
    #     torch.backends.cudnn.benchmark = False
    logging.info(f"Random seed set to {seed}")

set_seed(config.SEED)
logging.info(f"Using device: {config.DEVICE}")
logging.info(f"Pin memory: {config.PIN_MEMORY}")
logging.info(f"Number of workers for DataLoader: {config.NUM_WORKERS}")
logging.info(f"Using torch.compile: {config.COMPILE_MODEL} (PyTorch Version: {torch.__version__})")


# --- Google Drive Mounting (Simplified & Robust) ---
IS_DRIVE_MOUNTED = False
def mount_drive(mount_point='/content/drive'):
    global IS_DRIVE_MOUNTED
    if 'google.colab' in str(get_ipython()): # Check if in Colab
        try:
            from google.colab import drive
            if not os.path.exists(mount_point) or len(os.listdir(mount_point)) == 0 :
                 logging.info(f"Attempting to mount Google Drive at {mount_point}...")
                 drive.mount(mount_point, force_remount=True)
                 # Verify mount point access after mounting
                 if os.path.exists(mount_point) and os.access(mount_point, os.R_OK | os.W_OK):
                     logging.info(f"Google Drive mounted successfully and accessible at {mount_point}.")
                     IS_DRIVE_MOUNTED = True
                 else:
                     logging.error(f"Mounting seemed to succeed, but {mount_point} is not accessible/writable.")
                     IS_DRIVE_MOUNTED = False
            else:
                 logging.info(f"Google Drive already mounted at {mount_point}.")
                 IS_DRIVE_MOUNTED = True # Assume already mounted is accessible

        except ImportError:
            logging.warning("Running in Colab environment but 'google.colab.drive' not found.")
            IS_DRIVE_MOUNTED = False
        except Exception as e:
            logging.error(f"Error mounting Google Drive: {e}", exc_info=True)
            IS_DRIVE_MOUNTED = False
    else:
        logging.info("Not running in Google Colab environment. Skipping drive mount.")
        IS_DRIVE_MOUNTED = False # Assume not mounted if not Colab

mount_drive()

# --- Dataset Path Verification (Robust) ---
def verify_paths(data_path, preprocessed_path):
    """Checks if the main data path and expected subdirectories exist, and preprocessed dir is writable."""
    logging.info(f"Verifying dataset structure at: {data_path}")
    paths_ok = True
    needs_drive = data_path.startswith('/content/drive') or preprocessed_path.startswith('/content/drive')

    if needs_drive and not IS_DRIVE_MOUNTED:
        logging.error("Required paths are on Google Drive, but Drive is not mounted or accessible.")
        return False

    # Check Dataset Root
    if not data_path or not isinstance(data_path, str):
        logging.error("Config.DATASET_ROOT is not set or is not a string.")
        paths_ok = False
    elif not os.path.exists(data_path):
        logging.error(f"Dataset path does NOT exist: {data_path}")
        paths_ok = False
    elif not os.path.isdir(data_path):
        logging.error(f"Dataset path is NOT a directory: {data_path}")
        paths_ok = False
    else:
        logging.info(f"Dataset path exists and is a directory: {data_path}")
        # Check Subdirs
        for subdir in ["Au", "Tp"]:
            sub_path = os.path.join(data_path, subdir)
            if not os.path.exists(sub_path) or not os.path.isdir(sub_path):
                logging.error(f"Expected '{subdir}' subdirectory NOT found or is not a directory in: {data_path}")
                paths_ok = False
            else:
                 logging.info(f"Found '{subdir}' subdirectory.")

    # Check Preprocessed Dir Writable
    if not preprocessed_path or not isinstance(preprocessed_path, str):
         logging.error("Config.PREPROCESSED_DIR is not set or is not a string.")
         paths_ok = False
    else:
        try:
            os.makedirs(preprocessed_path, exist_ok=True)
            test_file = os.path.join(preprocessed_path, ".write_test")
            with open(test_file, "w") as f: f.write("test")
            os.remove(test_file)
            logging.info(f"Preprocessed directory is writable: {preprocessed_path}")
        except Exception as e:
            logging.error(f"Preprocessed directory check FAILED for '{preprocessed_path}': {e}")
            paths_ok = False

    if not paths_ok:
         logging.error("Path verification failed. Please check Config settings and Drive mount status.")
    return paths_ok

PATHS_VERIFIED = verify_paths(config.DATASET_ROOT, config.PREPROCESSED_DIR)


# --- Dataset Class (Mostly unchanged, PIL conversion is standard) ---
# (Keep the robust CASIADataset class from the original code here)
class CASIADataset(Dataset):
    # <<< PASTE the robust CASIADataset class definition from your original code here >>>
    # No major speed optimizations within the class itself are obvious,
    # the main data loading bottleneck is handled by DataLoader workers
    # and the preprocessing step.
    def __init__(self, images: np.ndarray, labels: np.ndarray, transforms: callable):
        if not isinstance(images, np.ndarray) or not isinstance(labels, np.ndarray):
             raise TypeError(f"Images and labels must be numpy arrays, got {type(images)} and {type(labels)}")
        if images.shape[0] != labels.shape[0]:
            raise ValueError(f"Number of images ({images.shape[0]}) and labels ({labels.shape[0]}) must match.")

        self.images = images # Should be uint8 (H, W, C)
        self.labels = labels # Should be int64
        self.transforms = transforms
        self.dataset_len = len(self.images)
        # <<< OPTIMIZATION: Pre-convert labels to tensors? Minor gain likely.
        # self.labels_tensor = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self):
        return self.dataset_len

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        if idx < 0 or idx >= self.dataset_len:
            raise IndexError(f"Index {idx} out of bounds for dataset of size {self.dataset_len}")

        image_np = self.images[idx]
        # label_val = self.labels[idx] # Original way
        label = torch.tensor(self.labels[idx], dtype=torch.long) # Direct conversion if not pre-converted

        # Basic check before converting to PIL
        if image_np.ndim != 3 or image_np.shape[2] != 3 or image_np.dtype != np.uint8:
            logging.error(f"Image at index {idx} has unexpected format: shape={image_np.shape}, dtype={image_np.dtype}. Returning dummy data.")
            # Return dummy data consistent with expected transform output type (float tensor)
            dummy_image = torch.zeros((3, config.IMG_SIZE, config.IMG_SIZE), dtype=torch.float32)
            # Return a label that can be filtered out later if needed
            dummy_label = torch.tensor(-1, dtype=torch.long)
            return dummy_image, dummy_label

        try:
            # Convert NumPy array [H, W, C] uint8 to PIL Image
            image = Image.fromarray(image_np)
            # Apply transforms (ToTensor, Normalize, Augmentations)
            image = self.transforms(image)
            # label = torch.tensor(label_val, dtype=torch.long) # Use pre-converted tensor if done in __init__
            return image, label
        except Exception as e:
            logging.error(f"Error processing image at index {idx}: {e}", exc_info=True)
            dummy_image = torch.zeros((3, config.IMG_SIZE, config.IMG_SIZE), dtype=torch.float32)
            dummy_label = torch.tensor(-1, dtype=torch.long)
            return dummy_image, dummy_label


# --- Data Loading and Preprocessing Function (More efficient npz check) ---
def load_or_preprocess_data(data_path: str, preprocessed_dir: str, img_size: int, dataset_fraction: float = 1.0, train_split: float = 0.7, val_split: float = 0.15, test_split: float = 0.15, seed: int = 42) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Loads preprocessed data if available, otherwise preprocesses, balances training set, and saves.
    Returns None for all arrays if a critical error occurs during loading/processing.
    """
    logging.info("--- Starting Data Loading/Preprocessing ---")
    if not PATHS_VERIFIED:
         logging.error("Path verification failed. Cannot proceed.")
         return None, None, None, None, None, None

    # Define file paths using an f-string for clarity
    split_str = f"train{int(train_split*100)}_val{int(val_split*100)}_test{int(test_split*100)}"
    frac_str = f"frac{int(dataset_fraction*100)}" if dataset_fraction < 1.0 else "full"
    base_filename = f"{split_str}_seed{seed}_img{img_size}_{frac_str}"
    train_file = os.path.join(preprocessed_dir, f"train_balanced_{base_filename}.npz")
    val_file = os.path.join(preprocessed_dir, f"val_{base_filename}.npz")
    test_file = os.path.join(preprocessed_dir, f"test_{base_filename}.npz")
    metadata_file = os.path.join(preprocessed_dir, f"metadata_{base_filename}.npz")

    # --- Attempt to Load Preprocessed Data ---
    required_files = [train_file, val_file, test_file, metadata_file]
    # <<< OPTIMIZATION: Check all files exist before trying to load any
    if all(os.path.exists(f) for f in required_files):
        logging.info("Found existing preprocessed files. Loading data...")
        try:
            # Load metadata first to verify settings
            with np.load(metadata_file, allow_pickle=False) as meta:
                 # Verify settings consistency
                m_img_size = int(meta['img_size'])
                m_seed = int(meta['seed'])
                m_fraction = float(meta.get('dataset_fraction', 1.0)) # Handle potential missing key

                if m_img_size != img_size or m_seed != seed or abs(m_fraction - dataset_fraction) > 1e-6:
                     logging.warning(f"Metadata mismatch! Found img={m_img_size}, seed={m_seed}, frac={m_fraction:.2f}. "
                                     f"Expected img={img_size}, seed={seed}, frac={dataset_fraction:.2f}. "
                                     "Reprocessing data is recommended.")
                     # Decide: Force reprocess? For now, log warning and continue loading.
                     # To force: raise FileNotFoundError("Metadata mismatch, forcing reprocess.")

            # Load actual data arrays
            # <<< OPTIMIZATION: Load into memory directly, assumes sufficient RAM.
            # If RAM is an issue, memory-mapping (mmap_mode='r') can be used, but might be slower access.
            with np.load(train_file, allow_pickle=False) as data:
                train_images, train_labels = data['images'], data['labels']
            with np.load(val_file, allow_pickle=False) as data:
                val_images, val_labels = data['images'], data['labels']
            with np.load(test_file, allow_pickle=False) as data:
                test_images, test_labels = data['images'], data['labels']

            logging.info("Loaded data shapes:")
            logging.info(f"  Train: Images={train_images.shape}, Labels={train_labels.shape}")
            logging.info(f"  Val:   Images={val_images.shape}, Labels={val_labels.shape}")
            logging.info(f"  Test:  Images={test_images.shape}, Labels={test_labels.shape}")

            # <<< OPTIMIZATION: Basic dtype check after loading
            if train_images.dtype != np.uint8 or val_images.dtype != np.uint8 or test_images.dtype != np.uint8:
                 logging.warning(f"Loaded images have unexpected dtype(s): "
                                 f"Train={train_images.dtype}, Val={val_images.dtype}, Test={test_images.dtype}. "
                                 f"Expected uint8. Trying to convert.")
                 try:
                     train_images = train_images.astype(np.uint8)
                     val_images = val_images.astype(np.uint8)
                     test_images = test_images.astype(np.uint8)
                 except Exception as e:
                     logging.error(f"Failed to convert loaded images to uint8: {e}. Cannot proceed.")
                     return None, None, None, None, None, None
            if train_labels.dtype != np.int64 or val_labels.dtype != np.int64 or test_labels.dtype != np.int64:
                 logging.warning(f"Loaded labels have unexpected dtype(s). Expected int64. Converting.")
                 train_labels = train_labels.astype(np.int64)
                 val_labels = val_labels.astype(np.int64)
                 test_labels = test_labels.astype(np.int64)

            logging.info("Successfully loaded and verified preprocessed data.")
            return train_images, train_labels, val_images, val_labels, test_images, test_labels

        except FileNotFoundError as e: # Catch the specific error if we force reprocess
             logging.warning(f"{e}")
             # Clean up potentially outdated/mismatched files before reprocessing
             for f in required_files:
                 if os.path.exists(f):
                     try: os.remove(f); logging.info(f"Removed outdated file: {f}")
                     except OSError: logging.warning(f"Could not remove potentially outdated file: {f}")

        except Exception as e:
            logging.error(f"Error loading preprocessed files from '{preprocessed_dir}': {e}. Attempting to re-process...", exc_info=True)
            # Clean up potentially corrupted files before reprocessing
            for f in required_files:
                 if os.path.exists(f):
                     try: os.remove(f); logging.info(f"Removed potentially corrupt file: {f}")
                     except OSError: logging.warning(f"Could not remove potentially corrupt file: {f}")

    # --- Preprocessing Steps (If loading failed or files didn't exist) ---
    logging.info(f"Preprocessing dataset from image files in '{data_path}'...")
    # (Keep the robust image loading loop from the original code here)
    # <<< PASTE the image loading, splitting, and balancing logic here >>>
    # This part is I/O heavy but only runs once if .npz files are saved correctly.
    # Ensure it saves images as np.uint8 and labels as np.int64
    # [...]
    # --- Start Copied Section ---
    authentic_dir = os.path.join(data_path, "Au")
    tampered_dir = os.path.join(data_path, "Tp")

    patterns = ['*.[jJ][pP][gG]', '*.[jJ][pP][eE][gG]', '*.[pP][nN][gG]', '*.[bB][mM][pP]', '*.[tT][iI][fF]', '*.[tT][iI][fF][fF]']
    authentic_paths = []
    for pattern in patterns: authentic_paths.extend(glob.glob(os.path.join(authentic_dir, pattern)))
    tampered_paths = []
    for pattern in patterns: tampered_paths.extend(glob.glob(os.path.join(tampered_dir, pattern)))

    num_auth_found = len(authentic_paths)
    num_tamp_found = len(tampered_paths)
    logging.info(f"Found {num_auth_found} potential authentic files and {num_tamp_found} potential tampered files.")

    if num_auth_found == 0 or num_tamp_found == 0:
        logging.error(f"Did not find images for both classes ('Au': {num_auth_found}, 'Tp': {num_tamp_found}). Check DATASET_ROOT and contents.")
        return None, None, None, None, None, None

    # Apply Dataset Fraction
    if dataset_fraction < 1.0:
        logging.info(f"Sampling {dataset_fraction*100:.1f}% of the found files (stratified).")
        paths = authentic_paths + tampered_paths
        labels_paths = [0] * num_auth_found + [1] * num_tamp_found
        paths_sampled, _, labels_sampled, _ = train_test_split(
            paths, labels_paths,
            train_size=dataset_fraction,
            stratify=labels_paths,
            random_state=seed
        )
        authentic_paths = [p for p, l in zip(paths_sampled, labels_sampled) if l == 0]
        tampered_paths = [p for p, l in zip(paths_sampled, labels_sampled) if l == 1]
        logging.info(f"Using {len(authentic_paths)} authentic and {len(tampered_paths)} tampered paths after sampling.")

    all_image_paths = authentic_paths + tampered_paths
    all_labels_for_paths = [0] * len(authentic_paths) + [1] * len(tampered_paths)

    images = []
    labels = []
    loaded_count = 0
    skipped_count = 0
    skipped_reasons: Dict[str, int] = {}

    logging.info(f"Attempting to load and resize {len(all_image_paths)} images to {img_size}x{img_size}...")
    progress_bar_load = tqdm(zip(all_image_paths, all_labels_for_paths), total=len(all_image_paths), desc="Loading & Resizing", dynamic_ncols=True)
    for img_path, label in progress_bar_load:
        try:
            progress_bar_load.set_description(f"Processing {os.path.basename(img_path)}")
            img = Image.open(img_path)
            if img.mode != 'RGB': img = img.convert('RGB')
            # <<< OPTIMIZATION: Consider Image.Resampling.BILINEAR if LANCZOS is too slow and quality is acceptable
            img = img.resize((img_size, img_size), Image.Resampling.LANCZOS)
            # <<< ENSURE uint8 FOR SAVING >>>
            img_np = np.array(img, dtype=np.uint8)

            if img_np.ndim != 3 or img_np.shape != (img_size, img_size, 3):
                reason = f"Unexpected shape {img_np.shape}"
                skipped_count += 1; skipped_reasons[reason] = skipped_reasons.get(reason, 0) + 1
                continue
            if img_np.size == 0:
                 reason = "Empty image data"
                 skipped_count += 1; skipped_reasons[reason] = skipped_reasons.get(reason, 0) + 1
                 continue

            images.append(img_np)
            # <<< ENSURE int64 FOR SAVING >>>
            labels.append(np.int64(label))
            loaded_count += 1

        except (IOError, OSError, Image.DecompressionBombError, SyntaxError) as e:
            reason = f"PIL/IO error: {type(e).__name__}"; skipped_count += 1; skipped_reasons[reason] = skipped_reasons.get(reason, 0) + 1
        except Exception as e:
            reason = f"Unexpected error: {type(e).__name__}"; logging.warning(f"Skipping {os.path.basename(img_path)} due to: {e}"); skipped_count += 1; skipped_reasons[reason] = skipped_reasons.get(reason, 0) + 1

    logging.info(f"Image loading finished. Loaded: {loaded_count}, Skipped: {skipped_count}")
    if skipped_count > 0: logging.warning(f"Skipped image reasons summary: {skipped_reasons}")
    if not images:
        logging.error("No images loaded successfully.")
        return None, None, None, None, None, None

    # <<< ENSURE CORRECT DTYPES BEFORE SPLIT >>>
    images = np.array(images, dtype=np.uint8)
    labels = np.array(labels, dtype=np.int64)

    actual_num_classes = len(np.unique(labels))
    if actual_num_classes < 2:
         logging.error(f"Only {actual_num_classes} class found after loading. Need at least 2.")
         return None, None, None, None, None, None
    if actual_num_classes != config.NUM_CLASSES:
         logging.warning(f"Found {actual_num_classes} unique labels, but config specified {config.NUM_CLASSES}. Check data/config.")
         # Consider adjusting config.NUM_CLASSES here if appropriate, but be careful.

    logging.info(f"Total images loaded into numpy array: {images.shape}. Labels: {labels.shape}")

    # Split into Train/Val/Test (Stratified)
    logging.info(f"Splitting data: Train={train_split}, Val={val_split}, Test={test_split} (Stratified)")
    try:
        remaining_images, test_images, remaining_labels, test_labels = train_test_split(
            images, labels, test_size=test_split, stratify=labels, random_state=seed)
        if train_split + val_split < 1e-6: # Avoid division by zero
             train_images_unbalanced, val_images, train_labels_unbalanced, val_labels = remaining_images, np.array([], dtype=images.dtype), remaining_labels, np.array([], dtype=labels.dtype)
        else:
             val_split_adjusted = val_split / (train_split + val_split)
             train_images_unbalanced, val_images, train_labels_unbalanced, val_labels = train_test_split(
                 remaining_images, remaining_labels, test_size=val_split_adjusted, stratify=remaining_labels, random_state=seed)
    except ValueError as e:
         logging.error(f"Error during train/val/test split (maybe too few samples per class?): {e}")
         return None, None, None, None, None, None

    logging.info("Dataset split (before balancing training set):")
    logging.info(f"  Train (unbalanced): {len(train_images_unbalanced)} images. Dist: {np.bincount(train_labels_unbalanced)}")
    logging.info(f"  Validation: {len(val_images)} images. Dist: {np.bincount(val_labels)}")
    logging.info(f"  Test: {len(test_images)} images. Dist: {np.bincount(test_labels)}")

    # Balance the Training Set via Oversampling
    logging.info("Balancing the training set using Oversampling...")
    train_images = train_images_unbalanced
    train_labels = train_labels_unbalanced
    unique_labels_train, counts_train = np.unique(train_labels_unbalanced, return_counts=True)

    if len(unique_labels_train) >= 2 and counts_train.min() > 0 and counts_train[0] != counts_train[1]:
        minority_label = unique_labels_train[np.argmin(counts_train)]
        n_minority = counts_train.min()
        n_majority = counts_train.max()
        n_to_add = n_majority - n_minority
        logging.info(f"  Oversampling minority class ({minority_label}) by adding {n_to_add} samples.")
        minority_indices = np.where(train_labels_unbalanced == minority_label)[0]
        np_rng = np.random.RandomState(seed)
        oversample_indices = np_rng.choice(minority_indices, size=n_to_add, replace=True)
        oversampled_images = train_images_unbalanced[oversample_indices]
        oversampled_labels = train_labels_unbalanced[oversample_indices]
        train_images = np.concatenate((train_images_unbalanced, oversampled_images), axis=0)
        train_labels = np.concatenate((train_labels_unbalanced, oversampled_labels), axis=0)
        train_images, train_labels = shuffle(train_images, train_labels, random_state=seed)
        logging.info(f"  Balanced train distribution: {np.bincount(train_labels)}")
    elif len(unique_labels_train) < 2:
         logging.warning("Training set contains only one class after splitting. Skipping balancing.")
    elif counts_train.min() == 0:
         logging.warning("Minority class has 0 samples in training set after split. Cannot balance.")
    else: # Already balanced or only one class
         logging.info("Training set already balanced or only contains one class.")

    # Save Processed Data
    logging.info(f"Saving preprocessed data to {preprocessed_dir}...")
    try:
        # Ensure correct dtypes are saved
        np.savez_compressed(train_file, images=train_images.astype(np.uint8), labels=train_labels.astype(np.int64))
        np.savez_compressed(val_file, images=val_images.astype(np.uint8), labels=val_labels.astype(np.int64))
        np.savez_compressed(test_file, images=test_images.astype(np.uint8), labels=test_labels.astype(np.int64))
        np.savez_compressed(metadata_file,
                            num_classes=config.NUM_CLASSES, img_size=img_size, seed=seed,
                            dataset_fraction=dataset_fraction, train_split=train_split,
                            val_split=val_split, test_split=test_split)
        logging.info("Successfully saved preprocessed data files.")
    except Exception as e:
        logging.error(f"Failed to save preprocessed data: {e}", exc_info=True)
        # Return in-memory data anyway, but log the failure
    # --- End Copied Section ---

    logging.info("--- Finished Data Loading/Preprocessing ---")
    # Return the potentially balanced train set and the val/test sets
    return train_images, train_labels, val_images, val_labels, test_images, test_labels


# --- Data Transformations ---
# <<< OPTIMIZATION: Consider slightly less aggressive augmentations if dataloading is the bottleneck
# For example, reduce rotation degrees, jitter intensity, or remove affine transform. Test impact.
train_transforms = transforms.Compose([
    # TrivialAugmentWide is often a good starting point, faster than manual combinations
    # transforms.TrivialAugmentWide(num_magnitude_bins=10), # Example alternative
    transforms.RandomResizedCrop(config.IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)), # Slightly less aggressive scale
    transforms.RandomRotation(degrees=10), # Slightly less rotation
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05), # Slightly less jitter
    transforms.RandomHorizontalFlip(p=0.5),
    # transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)), # Can be slow, consider removing if bottleneck
    transforms.ToTensor(), # Converts PIL image (H, W, C) [0, 255] to Tensor (C, H, W) [0.0, 1.0]
    transforms.Normalize(mean=config.MEAN, std=config.STD),
    # <<< OPTIMIZATION: Consider adding transforms.RandomErasing if needed for regularization
    # transforms.RandomErasing(p=0.2, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0),
])

val_test_transforms = transforms.Compose([
    # <<< OPTIMIZATION: Use BILINEAR for Resize if acceptable, it's faster than BICUBIC/LANCZOS
    transforms.Resize(config.IMG_SIZE, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(config.IMG_SIZE), # Ensure exact size after resize
    transforms.ToTensor(),
    transforms.Normalize(mean=config.MEAN, std=config.STD)
])
logging.info("Data transformations defined.")

# === Early Fusion Model (Mostly Unchanged) ===
# The model structure itself is standard. `timm` backbones are efficient.
# AdaptiveAvgPool is efficient. Convs/BN/ReLU are standard.
class EarlyFusionSwinEffNet(nn.Module):
    # <<< PASTE the EarlyFusionSwinEffNet class definition from your original code here >>>
    # Ensure it uses config parameters correctly.
    def __init__(self, num_classes, target_spatial_size=config.FUSION_TARGET_SPATIAL_SIZE,
                 fusion_channels=config.FUSION_CONV_CHANNELS, dropout_rate=config.FUSION_DROPOUT):
        super(EarlyFusionSwinEffNet, self).__init__()
        self.num_classes = num_classes
        self.target_spatial_size = target_spatial_size
        self.fusion_channels = fusion_channels
        self.dropout_rate = dropout_rate
        self.device = config.DEVICE # Use device from config

        logging.info("Initializing EarlyFusionSwinEffNet model...")
        logging.info(f" Fusion target spatial size: {self.target_spatial_size}")
        logging.info(f" Fusion conv channels: {self.fusion_channels}")
        logging.info(f" Fusion dropout rate: {self.dropout_rate}")

        # Load Backbones
        logging.info("Loading Swin Transformer (swin_tiny_patch4_window7_224) backbone...")
        self.swin = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, features_only=True, out_indices=(-1,))
        logging.info("Loading EfficientNet (efficientnet_b0) backbone...")
        self.efficientnet = timm.create_model('efficientnet_b0', pretrained=True, features_only=True, out_indices=(-1,))

        # Probe feature map dimensions
        swin_feat_dim, eff_feat_dim = self._probe_feature_dims()
        if swin_feat_dim is None or eff_feat_dim is None:
             raise RuntimeError("Could not probe feature dimensions from backbones.")
        logging.info(f"  - Detected Swin feature map channels: {swin_feat_dim}")
        logging.info(f"  - Detected EfficientNet feature map channels: {eff_feat_dim}")

        # Adaptive Pooling Layers
        self.swin_pool = nn.AdaptiveAvgPool2d((self.target_spatial_size, self.target_spatial_size))
        self.eff_pool = nn.AdaptiveAvgPool2d((self.target_spatial_size, self.target_spatial_size))

        # Fusion Convolution Block
        concat_channels = swin_feat_dim + eff_feat_dim
        logging.info(f"  - Concatenated feature channels: {concat_channels}")
        # <<< OPTIMIZATION: Use bias=False with BatchNorm layers, the bias is redundant
        self.fusion_conv1 = nn.Conv2d(concat_channels, self.fusion_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.fusion_bn1 = nn.BatchNorm2d(self.fusion_channels)
        self.fusion_relu = nn.ReLU(inplace=True) # Use inplace=True for minor memory saving

        # Final Classification Head
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten(start_dim=1)
        self.dropout = nn.Dropout(self.dropout_rate)
        self.classifier = nn.Linear(self.fusion_channels, self.num_classes)

        logging.info(f"  - Final classifier head: {self.fusion_channels} -> {self.num_classes}")

        # Note: Model moved to device later, before compilation or training starts
        self._initialize_weights() # Initialize weights of new layers
        logging.info("EarlyFusionSwinEffNet model initialization complete.")

    @torch.no_grad()
    def _probe_feature_dims(self):
        # Ensure model is on the correct device for probing
        probe_device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Use temp device for probing
        self.swin.to(probe_device)
        self.efficientnet.to(probe_device)
        self.eval()
        probe_input = torch.randn(2, 3, config.IMG_SIZE, config.IMG_SIZE).to(probe_device)
        swin_dim, eff_dim = None, None
        try:
            swin_features = self.swin(probe_input)
            if isinstance(swin_features, list) and len(swin_features) > 0: swin_dim = swin_features[0].shape[1]
            else: logging.error(f"Unexpected Swin output: {type(swin_features)}")

            eff_features = self.efficientnet(probe_input)
            if isinstance(eff_features, list) and len(eff_features) > 0: eff_dim = eff_features[0].shape[1]
            else: logging.error(f"Unexpected EffNet output: {type(eff_features)}")
        except Exception as e:
            logging.error(f"Error during feature dimension probing: {e}", exc_info=True)
        finally:
            # Move models back to CPU after probing if they weren't originally there
            # This is handled later when the whole model is moved to config.DEVICE
            self.train() # Set back to train mode
        return swin_dim, eff_dim

    def _initialize_weights(self):
        logging.info("Initializing weights for fusion and classifier layers...")
        for m in self.modules():
             # Check specifically for the layers we added (or check name)
             if isinstance(m, nn.Conv2d) and m is self.fusion_conv1:
                 nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                 if m.bias is not None: nn.init.constant_(m.bias, 0) # Though bias=False used
             elif isinstance(m, nn.BatchNorm2d) and m is self.fusion_bn1:
                 nn.init.constant_(m.weight, 1)
                 nn.init.constant_(m.bias, 0)
             elif isinstance(m, nn.Linear) and m is self.classifier:
                 nn.init.xavier_uniform_(m.weight)
                 if m.bias is not None: nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # No autocast needed here if used globally in training/validation loops
        swin_map = self.swin(x)[0]
        eff_map = self.efficientnet(x)[0]

        swin_pooled = self.swin_pool(swin_map)
        eff_pooled = self.eff_pool(eff_map)

        combined_features = torch.cat([swin_pooled, eff_pooled], dim=1)

        fused = self.fusion_conv1(combined_features)
        fused = self.fusion_bn1(fused)
        fused = self.fusion_relu(fused) # inplace=True helps memory slightly

        out = self.global_pool(fused)
        out = self.flatten(out)
        out = self.dropout(out)
        out = self.classifier(out)
        return out


# === Plotting Functions (Unchanged) ===
# (Keep the plot_training_history function)
def plot_training_history(history: Dict[str, List[float]], phase: int, model_name: str):
    """Plots training and validation loss and accuracy curves."""
    epochs = range(1, len(history.get('train_loss', [])) + 1)
    if not epochs:
        logging.warning(f"No history data found for phase {phase} to plot.")
        return

    plt.figure(figsize=(14, 6))
    # Loss
    plt.subplot(1, 2, 1)
    if 'train_loss' in history: plt.plot(epochs, history['train_loss'], 'bo-', label='Train Loss')
    if 'val_loss' in history: plt.plot(epochs, history['val_loss'], 'ro-', label='Validation Loss')
    plt.title(f'Phase {phase} Loss - {model_name}'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.ylim(bottom=0); plt.legend(); plt.grid(True, linestyle='--', alpha=0.6)
    # Accuracy
    plt.subplot(1, 2, 2)
    if 'train_acc' in history: plt.plot(epochs, history['train_acc'], 'bo-', label='Train Accuracy')
    if 'val_acc' in history: plt.plot(epochs, history['val_acc'], 'ro-', label='Validation Accuracy')
    plt.title(f'Phase {phase} Accuracy - {model_name}'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.ylim(0, 1); plt.legend(); plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    save_path = f'training_curves_phase_{phase}_{model_name}.png'
    try:
        plt.savefig(save_path); logging.info(f"Saved training curves plot: {save_path}")
    except Exception as e:
        logging.error(f"Failed to save training curves plot '{save_path}': {e}")
    plt.close()


# === Training & Validation Functions (Optimized) ===

# <<< OPTIMIZATION: Initialize GradScaler once globally
scaler = GradScaler(enabled=(config.DEVICE.type == 'cuda'))
logging.info(f"Gradient Scaler enabled: {scaler.is_enabled()}")

def train_one_epoch(model: nn.Module, dataloader: DataLoader, criterion: nn.Module, optimizer: optim.Optimizer, device: torch.device, scheduler: Optional[optim.lr_scheduler._LRScheduler], clip_value: float) -> Tuple[float, float]:
    """Trains the model for one epoch."""
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    start_time = time.time()

    # Use tqdm with leave=False for cleaner logs, especially when running multiple epochs
    progress_bar = tqdm(dataloader, desc="Training", leave=False, dynamic_ncols=True, unit="batch")
    for batch_idx, (inputs, labels) in enumerate(progress_bar):
        # Skip batches with dummy data (-1 label from CASIADataset error handling)
        valid_indices = labels != -1
        if not valid_indices.all():
            inputs = inputs[valid_indices]
            labels = labels[valid_indices]
            if inputs.size(0) == 0:
                 logging.warning(f"Skipping batch {batch_idx+1}/{len(dataloader)} due to all dummy data.")
                 continue

        # <<< OPTIMIZATION: Use non_blocking=True for potential async transfer
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        # <<< OPTIMIZATION: More efficient zeroing
        optimizer.zero_grad(set_to_none=True)

        # <<< OPTIMIZATION: Automatic Mixed Precision context
        with autocast(enabled=scaler.is_enabled()):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        if torch.isnan(loss):
            logging.error(f"NaN loss detected at batch {batch_idx+1}. Skipping batch update.")
            continue # Skip optimizer step if loss is NaN

        # Scale loss, backward pass, and optimizer step
        scaler.scale(loss).backward()

        # Optional: Unscale gradients before clipping if needed by the clipping method or for inspection
        # scaler.unscale_(optimizer) # Often not strictly necessary if clip_grad_norm_ handles scaled grads

        # <<< OPTIMIZATION: Clip gradients (applied to parameters directly)
        # Clip *after* scaling, *before* optimizer step. Scaler handles unscaling internally during step().
        # Note: clip_grad_norm_ works on unscaled gradients. If scaler.unscale_ isn't called,
        # you might clip scaled gradients. This is usually fine, but check if behavior is expected.
        # A common practice is scaler.unscale_ *before* clipping if precise unscaled norm is needed.
        if clip_value > 0:
             scaler.unscale_(optimizer) # Unscale first for accurate norm calculation
             nn.utils.clip_grad_norm_(model.parameters(), clip_value)

        scaler.step(optimizer)
        scaler.update()

        # Step the scheduler (if one is provided and it's step-based)
        # Example: if isinstance(scheduler, optim.lr_scheduler.CosineAnnealingLR): scheduler.step()
        # This depends on how the scheduler is configured (per step or per epoch)
        # For CosineAnnealingLR with T_max=total_steps, step per iteration.
        if scheduler and isinstance(scheduler, (optim.lr_scheduler.CosineAnnealingLR, optim.lr_scheduler.OneCycleLR)):
              scheduler.step()

        # Accumulate Metrics
        running_loss += loss.item() * inputs.size(0) # loss.item() is average loss in batch
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        # Update progress bar less frequently or with smoothed values if it causes overhead
        if batch_idx % 20 == 0: # Update postfix less often
            current_lr = optimizer.param_groups[0]['lr']
            mem_allocated = torch.cuda.memory_allocated(device)/1e9 if device.type=='cuda' else 0
            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}", # Show current batch loss
                acc=f"{correct_predictions/total_samples:.4f}",
                lr=f"{current_lr:.1e}",
                mem=f"{mem_allocated:.2f}G"
            )

    epoch_duration = time.time() - start_time
    if total_samples == 0:
        logging.warning("Training epoch finished but no valid samples processed.")
        return 0.0, 0.0

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_predictions / total_samples
    return epoch_loss, epoch_acc


@torch.no_grad() # Essential for validation efficiency
def validate(model: nn.Module, dataloader: DataLoader, criterion: nn.Module, device: torch.device) -> Tuple[float, float]:
    """Validates the model on the validation set."""
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    all_preds_val = []
    all_labels_val = []

    progress_bar = tqdm(dataloader, desc="Validation", leave=False, dynamic_ncols=True, unit="batch")
    for inputs, labels in progress_bar:
        # Filter dummy data
        valid_indices = labels != -1
        if not valid_indices.all():
            inputs = inputs[valid_indices]
            labels = labels[valid_indices]
            if inputs.size(0) == 0: continue

        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        # Use autocast for potentially faster inference, even without GradScaler
        with autocast(enabled=(device.type == 'cuda')):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        # Collect preds/labels for report (consider doing this only at the end if memory is tight)
        all_preds_val.extend(predicted.cpu().numpy())
        all_labels_val.extend(labels.cpu().numpy())

        # Update progress bar postfix less frequently
        if len(progress_bar) % 10 == 0:
            mem_allocated = torch.cuda.memory_allocated(device)/1e9 if device.type=='cuda' else 0
            progress_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct_predictions/total_samples:.4f}", mem=f"{mem_allocated:.2f}G")


    if total_samples == 0:
        logging.warning("Validation finished but no valid samples processed.")
        return float('inf'), 0.0 # Return high loss, zero accuracy

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_predictions / total_samples

    # Optionally generate report here, or return preds/labels to generate outside
    # if all_labels_val and all_preds_val:
    #     try:
    #         # logging.info("\n--- Validation Set Report (End of Epoch) ---")
    #         report = classification_report(all_labels_val, all_preds_val, target_names=['Authentic', 'Tampered'], digits=4, zero_division=0)
    #         # logging.info("\n" + report)
    #         # logging.info("--- End Validation Set Report ---")
    #     except Exception as e:
    #         logging.error(f"Could not generate validation classification report: {e}")

    return epoch_loss, epoch_acc


def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, criterion: nn.Module, optimizer: optim.Optimizer, scheduler: Optional[optim.lr_scheduler._LRScheduler], num_epochs: int, device: torch.device, patience: int, clip_value: float, phase: int, model_save_name: str) -> Tuple[nn.Module, Dict[str, List[float]]]:
    """Trains the model for a given phase with early stopping."""
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_epoch = -1

    logging.info(f"--- Starting Training Phase {phase} for {num_epochs} Epochs ---")
    logging.info(f" Model will be saved to: {model_save_name}")
    logging.info(f" Optimizer: {type(optimizer).__name__}, LR: {optimizer.param_groups[0]['lr']:.1e}")
    logging.info(f" Scheduler: {type(scheduler).__name__ if scheduler else 'None'}")
    logging.info(f" Early stopping patience: {patience}")
    logging.info(f" Gradient clipping value: {clip_value if clip_value > 0 else 'Disabled'}")

    total_start_time = time.time()

    for epoch in range(num_epochs):
        epoch_num = epoch + 1
        epoch_start_time = time.time()
        logging.info(f"===> Epoch {epoch_num}/{num_epochs}, Phase {phase} <===")

        # Training
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scheduler, clip_value)

        # Validation
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        # Update history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        epoch_duration = time.time() - epoch_start_time
        current_lr = optimizer.param_groups[0]['lr']

        log_msg = (f"Epoch {epoch_num} Summary: "
                   f"Time={epoch_duration:.2f}s, LR={current_lr:.2e}, "
                   f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
                   f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")
        logging.info(log_msg)

        # Step LR scheduler if it's epoch-based (e.g., StepLR, MultiStepLR)
        if scheduler and not isinstance(scheduler, (optim.lr_scheduler.CosineAnnealingLR, optim.lr_scheduler.OneCycleLR)):
              scheduler.step() # Or scheduler.step(val_loss) for ReduceLROnPlateau

        # Checkpoint Saving & Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_epoch = epoch_num
            try:
                # <<< OPTIMIZATION: Save only state_dict unless whole object needed
                torch.save(model.state_dict(), model_save_name)
                logging.info(f"Val loss improved to {best_val_loss:.4f}. Model state saved.")
            except Exception as e:
                logging.error(f"Error saving model checkpoint: {e}")
        else:
            patience_counter += 1
            logging.info(f"Val loss ({val_loss:.4f}) did not improve from best ({best_val_loss:.4f} @ epoch {best_epoch}). Patience: {patience_counter}/{patience}")
            if patience_counter >= patience:
                logging.warning(f"Early stopping triggered after {epoch_num} epochs.")
                break

        # Optional: Clear CUDA cache periodically if memory pressure is high, but can hide underlying issues
        # if device.type == 'cuda' and epoch_num % 5 == 0: torch.cuda.empty_cache()

    total_training_time = time.time() - total_start_time
    logging.info(f"--- Finished Training Phase {phase}. Total Time: {total_training_time:.2f}s ---")
    logging.info(f"Best validation loss: {best_val_loss:.4f} at epoch {best_epoch}")

    # Load Best Model state found during this phase
    if os.path.exists(model_save_name):
        try:
            logging.info(f"Loading best model state from phase {phase} ('{os.path.basename(model_save_name)}')")
            # <<< Ensure map_location is set correctly if loading on a different device type
            model.load_state_dict(torch.load(model_save_name, map_location=device), strict=True)
        except Exception as e:
            logging.error(f"Error loading best model state '{model_save_name}': {e}. Using model state from last epoch.")
    else:
        logging.warning(f"Best model file '{model_save_name}' not found after phase {phase}. Using model state from last epoch.")

    return model, history

# === Evaluation Function (Optimized) ===
@torch.no_grad()
def evaluate_model(model: nn.Module, test_loader: DataLoader, device: torch.device, num_classes: int, model_name: str) -> None:
    """Evaluates the final model on the test set."""
    logging.info("\n--- Starting Final Evaluation on Test Set ---")
    model.eval()

    all_preds = []
    all_labels = []
    total_samples = 0
    start_time = time.time()

    progress_bar = tqdm(test_loader, desc="Testing", leave=False, dynamic_ncols=True, unit="batch")
    for inputs, labels in progress_bar:
        # Filter dummy data
        valid_indices = labels != -1
        if not valid_indices.all():
            inputs = inputs[valid_indices]
            labels = labels[valid_indices]
            if inputs.size(0) == 0: continue

        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        # <<< OPTIMIZATION: Use autocast during evaluation too
        with autocast(enabled=(device.type == 'cuda')):
             outputs = model(inputs)

        # Get predictions (no need for loss calculation)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total_samples += inputs.size(0)

        if len(progress_bar) % 10 == 0:
            mem_allocated = torch.cuda.memory_allocated(device)/1e9 if device.type=='cuda' else 0
            progress_bar.set_postfix(mem=f"{mem_allocated:.2f}G")


    eval_duration = time.time() - start_time
    logging.info(f"Test set evaluation finished in {eval_duration:.2f}s on {total_samples} samples.")

    if total_samples == 0 or not all_labels or not all_preds:
         logging.error("No valid samples processed or predictions collected during testing. Cannot evaluate.")
         return

    # Metrics Calculation & Display (Keep the original robust plotting/reporting)
    logging.info("\n--- Test Set Results ---")
    target_names = ['Authentic (0)', 'Tampered (1)'] if num_classes == 2 else [f'Class_{i}' for i in range(num_classes)]

    try:
        report = classification_report(all_labels, all_preds, target_names=target_names, digits=4, zero_division=0)
        logging.info("Classification Report:\n" + report)
        print("\nTest Set Classification Report:\n", report) # Print to console too
    except Exception as e:
        logging.error(f"Could not generate classification report: {e}")

    # Confusion Matrix (Counts)
    try:
        cm_counts = confusion_matrix(all_labels, all_preds)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm_counts, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names, annot_kws={"size": 12})
        plt.xlabel("Predicted Labels"); plt.ylabel("True Labels"); plt.title(f"Test Confusion Matrix (Counts) - {model_name}"); plt.tight_layout()
        save_path_counts = f'test_confusion_matrix_counts_{model_name}.png'; plt.savefig(save_path_counts)
        logging.info(f"Saved confusion matrix (counts): {save_path_counts}"); plt.close()
    except Exception as e:
        logging.error(f"Could not generate/save confusion matrix (counts): {e}")

    # Confusion Matrix (Percentage)
    try:
        cm_percent = confusion_matrix(all_labels, all_preds, normalize='true')
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm_percent, annot=True, fmt=".2%", cmap="Greens", xticklabels=target_names, yticklabels=target_names, annot_kws={"size": 12})
        plt.xlabel("Predicted Labels"); plt.ylabel("True Labels"); plt.title(f"Test Confusion Matrix (%) - {model_name}"); plt.tight_layout()
        save_path_percent = f'test_confusion_matrix_percent_{model_name}.png'; plt.savefig(save_path_percent)
        logging.info(f"Saved confusion matrix (percentage): {save_path_percent}"); plt.close()
    except Exception as e:
        logging.error(f"Could not generate/save confusion matrix (percentage): {e}")

    logging.info("--- Finished Final Evaluation ---")


# === Main Execution Block ===
if __name__ == "__main__":

    logging.info("="*10 + " Starting Optimized Early Fusion Experiment Run " + "="*10)
    logging.info(f"Model Name: {config.MODEL_NAME}")
    logging.info(f"PyTorch Version: {torch.__version__}")
    logging.info(f"Using Device: {config.DEVICE}")
    logging.info(f"Batch Size: {config.BATCH_SIZE}, Num Workers: {config.NUM_WORKERS}")
    logging.info(f"Use torch.compile: {config.COMPILE_MODEL}")

    if not PATHS_VERIFIED:
        logging.critical("Dataset root path or preprocessed dir verification failed. Exiting.")
        exit(1)

    # Load or Preprocess Data
    train_images, train_labels, val_images, val_labels, test_images, test_labels = load_or_preprocess_data(
        data_path=config.DATASET_ROOT,
        preprocessed_dir=config.PREPROCESSED_DIR,
        img_size=config.IMG_SIZE,
        dataset_fraction=config.DATASET_FRACTION,
        # Use splits from config if defined, otherwise use defaults
        train_split=getattr(config, 'TRAIN_SIZE', 0.7),
        val_split=getattr(config, 'VAL_SIZE', 0.15),
        test_split=getattr(config, 'TEST_SIZE', 0.15),
        seed=config.SEED
    )

    if train_images is None or val_images is None or test_images is None:
         logging.critical("Data loading/preprocessing failed. Exiting.")
         exit(1)

    # Create Datasets and DataLoaders
    try:
        logging.info("Creating Datasets...")
        train_dataset = CASIADataset(train_images, train_labels, train_transforms)
        val_dataset = CASIADataset(val_images, val_labels, val_test_transforms)
        test_dataset = CASIADataset(test_images, test_labels, val_test_transforms)
        logging.info(f"Dataset sizes: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")

        # Clear large numpy arrays from memory after creating datasets if RAM is tight
        del train_images, train_labels, val_images, val_labels, test_images, test_labels
        import gc
        gc.collect()
        logging.info("Cleaned up large NumPy arrays from memory.")

        logging.info("Creating DataLoaders...")
        # <<< OPTIMIZATION: Use config values for workers, pin_memory, batch_size
        # <<< OPTIMIZATION: persistent_workers=True can speed up epoch starts if num_workers > 0
        use_persistent_workers = config.NUM_WORKERS > 0

        train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                                  num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY,
                                  drop_last=True, # Good for consistent batch sizes
                                  persistent_workers=use_persistent_workers)
        val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                                num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY,
                                persistent_workers=use_persistent_workers)
        test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                                 num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY,
                                 persistent_workers=use_persistent_workers)

        logging.info(f"DataLoaders created (persistent_workers={use_persistent_workers}):")
        if len(train_loader) == 0 or len(val_loader) == 0 or len(test_loader) == 0:
            logging.critical("One or more DataLoaders are empty! Check data/batch size. Exiting.")
            exit(1)
        logging.info(f"  Train batches: {len(train_loader)}")
        logging.info(f"  Val batches: {len(val_loader)}")
        logging.info(f"  Test batches: {len(test_loader)}")

    except Exception as e:
        logging.critical(f"Error creating Datasets or DataLoaders: {e}", exc_info=True)
        exit(1)


    # Initialize Model, Loss, Optimizers
    model = None
    try:
        logging.info("Initializing Model and Loss...")
        model = EarlyFusionSwinEffNet(num_classes=config.NUM_CLASSES)
        model.to(config.DEVICE) # Move model to target device

        # <<< OPTIMIZATION: Apply torch.compile if enabled and available
        if config.COMPILE_MODEL:
            try:
                logging.info("Attempting to compile model with torch.compile()...")
                # Common modes: 'default', 'reduce-overhead', 'max-autotune'
                # 'reduce-overhead' good for small models/batches, 'max-autotune' takes longer to compile but might be faster runtime
                model = torch.compile(model, mode='default')
                logging.info("Model compilation successful.")
            except Exception as e:
                logging.warning(f"torch.compile failed: {e}. Proceeding without compilation.")
                config.COMPILE_MODEL = False # Disable flag if compilation fails

        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        logging.info(f"Model '{config.MODEL_NAME}' initialized on {config.DEVICE}.")
        logging.info(f"  Total Parameters: {total_params:,}")
        logging.info(f"  Initially Trainable Parameters: {trainable_params:,}")

        criterion = nn.CrossEntropyLoss()
        logging.info(f"Loss function ({type(criterion).__name__}) initialized.")

    except Exception as e:
        logging.critical(f"Error initializing model or loss function: {e}", exc_info=True)
        exit(1)


    # --- Training Phase 1: Train Fusion/Classifier Layers ---
    history_phase1 = None
    if config.EPOCHS_PHASE1 > 0:
        try:
            logging.info("\n" + "="*10 + f" Phase 1: Training Fusion/Classifier ({config.EPOCHS_PHASE1} Epochs) " + "="*10)
            model_save_name_phase1 = f"best_model_{config.MODEL_NAME}_phase1.pth"

            # Freeze backbones
            num_frozen = 0
            for name, param in model.named_parameters():
                # Handle potential torch.compile wrapper (_orig_mod)
                prefix = name.split('.')[0] if not name.startswith('_orig_mod.') else name.split('.')[1]
                if prefix in ['swin', 'efficientnet']:
                    param.requires_grad = False
                    num_frozen += param.numel()
                else:
                    param.requires_grad = True

            params_to_optimize_p1 = [p for p in model.parameters() if p.requires_grad]
            num_trainable_p1 = sum(p.numel() for p in params_to_optimize_p1)
            logging.info(f"Phase 1 - Frozen {num_frozen:,} backbone parameters.")
            logging.info(f"Phase 1 - Optimizing {num_trainable_p1:,} fusion/classifier parameters.")

            # Optimizer for Phase 1
            # <<< OPTIMIZATION: Consider fused optimizers if available (e.g., from Apex or native torch)
            # Needs testing, AdamW is generally safe and performant.
            optimizer_p1 = optim.AdamW(params_to_optimize_p1, lr=config.LEARNING_RATE_PHASE1, weight_decay=config.WEIGHT_DECAY)

            # Scheduler for Phase 1 (Cosine Annealing per step)
            total_steps_p1 = len(train_loader) * config.EPOCHS_PHASE1
            # <<< OPTIMIZATION: Ensure eta_min is very small
            scheduler_p1 = optim.lr_scheduler.CosineAnnealingLR(optimizer_p1, T_max=total_steps_p1, eta_min=1e-8)
            logging.info(f"Phase 1 - Cosine LR Scheduler: T_max={total_steps_p1} steps, eta_min={scheduler_p1.eta_min:.1e}")

            # Train
            model, history_phase1 = train_model(
                model=model, train_loader=train_loader, val_loader=val_loader,
                criterion=criterion, optimizer=optimizer_p1, scheduler=scheduler_p1,
                num_epochs=config.EPOCHS_PHASE1, device=config.DEVICE,
                patience=config.EARLY_STOPPING_PATIENCE, clip_value=config.CLIP_GRAD_NORM,
                phase=1, model_save_name=model_save_name_phase1
            )
            if history_phase1: plot_training_history(history_phase1, 1, config.MODEL_NAME)

        except Exception as e:
            logging.critical(f"Error during Training Phase 1: {e}", exc_info=True)
            exit(1)
    else:
        logging.info("Skipping Training Phase 1 (EPOCHS_PHASE1 = 0).")


    # --- Training Phase 2: Fine-tune Entire Model ---
    history_phase2 = None
    if config.EPOCHS_PHASE2 > 0:
        try:
            logging.info("\n" + "="*10 + f" Phase 2: Fine-tuning Entire Model ({config.EPOCHS_PHASE2} Epochs) " + "="*10)
            model_save_name_phase2 = f"best_model_{config.MODEL_NAME}_phase2_final.pth"

            # Unfreeze all parameters
            num_unfrozen = 0
            for param in model.parameters():
                if not param.requires_grad:
                     param.requires_grad = True
                     num_unfrozen += 1
            logging.info(f"Phase 2 - Unfrozen {num_unfrozen:,} parameters.")
            params_to_optimize_p2 = model.parameters()
            num_trainable_p2 = sum(p.numel() for p in params_to_optimize_p2)
            logging.info(f"Phase 2 - Optimizing all {num_trainable_p2:,} parameters.")

            # Optimizer for Phase 2 (Lower LR)
            optimizer_p2 = optim.AdamW(params_to_optimize_p2, lr=config.LEARNING_RATE_PHASE2, weight_decay=config.WEIGHT_DECAY)

            # Scheduler for Phase 2
            total_steps_p2 = len(train_loader) * config.EPOCHS_PHASE2
            scheduler_p2 = optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=total_steps_p2, eta_min=1e-9) # Even smaller eta_min
            logging.info(f"Phase 2 - Cosine LR Scheduler: T_max={total_steps_p2} steps, eta_min={scheduler_p2.eta_min:.1e}")

            # Train
            model, history_phase2 = train_model(
                model=model, train_loader=train_loader, val_loader=val_loader,
                criterion=criterion, optimizer=optimizer_p2, scheduler=scheduler_p2,
                num_epochs=config.EPOCHS_PHASE2, device=config.DEVICE,
                patience=config.EARLY_STOPPING_PATIENCE, clip_value=config.CLIP_GRAD_NORM,
                phase=2, model_save_name=model_save_name_phase2
            )
            if history_phase2: plot_training_history(history_phase2, 2, config.MODEL_NAME)

        except Exception as e:
            logging.critical(f"Error during Training Phase 2: {e}", exc_info=True)
            # Consider whether to evaluate the best model from phase 1 or exit
            exit(1)
    else:
        logging.info("Skipping Training Phase 2 (EPOCHS_PHASE2 = 0).")


    # --- Final Evaluation on Test Set ---
    try:
        logging.info("\n" + "="*10 + " Final Evaluation on Test Set " + "="*10)
        # Ensure the *best* model overall is loaded for evaluation.
        # If phase 2 ran, its best model was loaded by train_model.
        # If only phase 1 ran, its best model should be loaded.
        final_model_path = f"best_model_{config.MODEL_NAME}_phase2_final.pth" if config.EPOCHS_PHASE2 > 0 else f"best_model_{config.MODEL_NAME}_phase1.pth"
        if os.path.exists(final_model_path):
             logging.info(f"Loading final best model state for evaluation from: '{os.path.basename(final_model_path)}'")
             try:
                 # Handle potential torch.compile state dict differences if necessary
                 # Usually loading state_dict works fine, but sometimes requires unwrapping.
                 state_dict = torch.load(final_model_path, map_location=config.DEVICE)
                 # If model was compiled, state_dict might be from _orig_mod. Adapt loading if needed.
                 # For simple cases:
                 # if config.COMPILE_MODEL and not list(state_dict.keys())[0].startswith('_orig_mod.'):
                 #    state_dict = {'_orig_mod.'+k: v for k, v in state_dict.items()} # Example adaptation
                 model.load_state_dict(state_dict, strict=True)
             except Exception as e:
                 logging.error(f"FAILED to load best model state from {final_model_path}: {e}. Evaluation might use last epoch's state.", exc_info=True)
        else:
             logging.warning(f"Final best model file '{final_model_path}' not found. Evaluating model state currently in memory (likely last epoch of last phase).")


        evaluate_model(
            model=model,
            test_loader=test_loader,
            device=config.DEVICE,
            num_classes=config.NUM_CLASSES,
            model_name=config.MODEL_NAME
        )

    except Exception as e:
         logging.critical(f"Error during final evaluation: {e}", exc_info=True)


    logging.info("="*10 + " Optimized Early Fusion Experiment Run Finished " + "="*10)
    logging.info(f"Log file saved to: {log_file}")

Mounted at /content/drive


<ipython-input-2-b926c4a3dd64>:693: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(config.DEVICE.type == 'cuda'))
Processing Tp_D_CRD_M_B_nat00032_nat00033_10077.tif: 100%|██████████| 11547/11547 [22:18<00:00,  8.63it/s]
Training:   0%|          | 0/163 [00:00<?, ?batch/s]<ipython-input-2-b926c4a3dd64>:723: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):
Training:   1%|          | 1/163 [02:03<5:34:00, 123.71s/batch, acc=0.5312, loss=0.7834, lr=1.0e-04, mem=0.19G]<ipython-input-2-b926c4a3dd64>:723: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):
Validation:   0%|          | 0/28 [00:00<?, ?batch/s]<ipython-input-2-b926c4a3dd64>:805: Future


Test Set Classification Report:
                precision    recall  f1-score   support

Authentic (0)     0.8146    0.5943    0.6872      1124
 Tampered (1)     0.5005    0.7504    0.6005       609

     accuracy                         0.6492      1733
    macro avg     0.6576    0.6724    0.6439      1733
 weighted avg     0.7043    0.6492    0.6568      1733



In [ ]:
# --- PARTIE 1: INSTALLATION DES LIBRAIRIES ---
!pip install -q thop timm torchmetrics scikit-learn

# --- PARTIE 2: IMPORTS ---
from google.colab import drive
import os, glob, logging, random, numpy as np, torch, timm
from PIL import Image
from torchvision import transforms
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchmetrics.classification import MulticlassF1Score
from sklearn.model_selection import train_test_split
from torch.amp import GradScaler, autocast
from thop import profile
from tqdm import tqdm

# --- MONTAGE DRIVE ---
print("Montage de Google Drive...")
drive.mount('/content/drive', force_remount=True)

# --- CONFIGURATION ---
class Config:
    EPOCHS_PHASE1 = 2
    DATASET_FRACTION = 0.2
    IMG_SIZE = 224
    BATCH_SIZE = 64
    NUM_CLASSES = 2
    PREPROCESSED_DIR = "/content/drive/MyDrive/preprocessed_casia_earlyfusion_optimized_v6"
    DATA_PATH = "/content/drive/MyDrive/CASIA2.0_revised_corrected/casia"
    MODEL_NAME = "early_fusion_swin_effnet_optimized_v6"
    SEED = 42
    TRAIN_SPLIT = 0.7
    VAL_SPLIT = 0.15
    TEST_SPLIT = 0.15
    MEAN = [0.485, 0.456, 0.406]
    STD = [0.229, 0.224, 0.225]
    FUSION_TARGET_SPATIAL_SIZE = 7
    FUSION_CONV_CHANNELS = 512
    FUSION_DROPOUT = 0.3
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = Config()

# --- FIXER LE HASARD ---
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.SEED)

# --- DATASET ---
class CASIADataset(Dataset):
    def __init__(self, images, labels, transforms):
        self.images = images
        self.labels = labels
        self.transforms = transforms

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        img = Image.fromarray((image * 255).astype(np.uint8)) if image.max() <= 1.0 else Image.fromarray(image.astype(np.uint8))
        return self.transforms(img.convert('RGB')), torch.tensor(label, dtype=torch.long)

# --- CHARGEMENT OU PRÉTRAITEMENT DES DONNÉES ---
def load_or_preprocess_data(data_path, preprocessed_dir, img_size, dataset_fraction, train_split, val_split, test_split, seed):
    base_fn = f"train{int(train_split*100)}_val{int(val_split*100)}_test{int(test_split*100)}_seed{seed}"
    train_f = os.path.join(preprocessed_dir, f"train_{base_fn}.npz")
    val_f = os.path.join(preprocessed_dir, f"val_{base_fn}.npz")
    test_f = os.path.join(preprocessed_dir, f"test_{base_fn}.npz")

    if os.path.exists(train_f) and os.path.exists(val_f) and os.path.exists(test_f):
        with np.load(train_f) as d: t_img, t_lbl = d['images'], d['labels']
        with np.load(val_f) as d: v_img, v_lbl = d['images'], d['labels']
        with np.load(test_f) as d: ts_img, ts_lbl = d['images'], d['labels']
    else:
        os.makedirs(preprocessed_dir, exist_ok=True)
        auth_p = sorted(glob.glob(os.path.join(data_path, "Au", "*.*")))
        tamp_p = sorted(glob.glob(os.path.join(data_path, "Tp", "*.*")))
        imgs, lbls = [], []

        def process(paths, label):
            for p in tqdm(paths, desc=f"Processing {label}"):
                try:
                    img = Image.open(p).convert('RGB').resize((img_size, img_size))
                    imgs.append(np.array(img))
                    lbls.append(label)
                except: continue

        process(auth_p, 0)
        process(tamp_p, 1)

        imgs, lbls = np.array(imgs), np.array(lbls)
        idxs = np.arange(len(imgs))
        train_val_idx, test_idx = train_test_split(idxs, test_size=test_split, random_state=seed, stratify=lbls)
        train_idx, val_idx = train_test_split(train_val_idx, test_size=val_split/(train_split+val_split), random_state=seed, stratify=lbls[train_val_idx])

        t_img, t_lbl = imgs[train_idx], lbls[train_idx]
        v_img, v_lbl = imgs[val_idx], lbls[val_idx]
        ts_img, ts_lbl = imgs[test_idx], lbls[test_idx]

        np.savez_compressed(train_f, images=t_img, labels=t_lbl)
        np.savez_compressed(val_f, images=v_img, labels=v_lbl)
        np.savez_compressed(test_f, images=ts_img, labels=ts_lbl)

    def fraction(i, l): num = int(len(i) * dataset_fraction); return i[:num], l[:num]
    if dataset_fraction < 1.0:
        t_img, t_lbl = fraction(t_img, t_lbl)
        v_img, v_lbl = fraction(v_img, v_lbl)
        ts_img, ts_lbl = fraction(ts_img, ts_lbl)

    return t_img, t_lbl, v_img, v_lbl, ts_img, ts_lbl

# --- MODÈLE EARLY FUSION ---
class EarlyFusionSwinEffNet(nn.Module):
    def __init__(self, num_classes, fusion_target_spatial_size, fusion_conv_channels, fusion_dropout):
        super().__init__()

        self.swin = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, features_only=True, out_indices=(-1,))
        self.efficientnet = timm.create_model('efficientnet_b0', pretrained=True, features_only=True, out_indices=(-1,))

        # Pool pour mise à l'échelle spatiale
        self.swin_pool = nn.AdaptiveAvgPool2d((fusion_target_spatial_size, fusion_target_spatial_size))
        self.eff_pool = nn.AdaptiveAvgPool2d((fusion_target_spatial_size, fusion_target_spatial_size))

        # Initialisation à None, on définira dynamiquement après le premier passage
        self.fusion_conv = None
        self.fusion_bn = None
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(fusion_dropout)
        self.classifier = nn.Linear(fusion_conv_channels, num_classes)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fusion_conv_channels = fusion_conv_channels

    def forward(self, x):
        with torch.no_grad():
            swin_feat = self.swin(x)[0]
            eff_feat = self.efficientnet(x)[0]

        swin_feat = self.swin_pool(swin_feat)
        eff_feat = self.eff_pool(eff_feat)

        fused = torch.cat([swin_feat, eff_feat], dim=1)

        # Création dynamique du conv si pas encore défini
        if self.fusion_conv is None:
            in_channels = fused.shape[1]
            self.fusion_conv = nn.Conv2d(in_channels, self.fusion_conv_channels, kernel_size=1, bias=False).to(fused.device)
            self.fusion_bn = nn.BatchNorm2d(self.fusion_conv_channels).to(fused.device)

        x = self.fusion_conv(fused)
        x = self.fusion_bn(x)
        x = self.relu(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.classifier(x)


# --- PARTIE 3 : ENTRAÎNEMENT & ANALYSE ---
try:
    train_img, train_lbl, _, _, test_img, test_lbl = load_or_preprocess_data(
        data_path=config.DATA_PATH,
        preprocessed_dir=config.PREPROCESSED_DIR,
        img_size=config.IMG_SIZE,
        dataset_fraction=config.DATASET_FRACTION,
        train_split=config.TRAIN_SPLIT,
        val_split=config.VAL_SPLIT,
        test_split=config.TEST_SPLIT,
        seed=config.SEED
    )
    transform = transforms.Compose([
        transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(config.MEAN, config.STD)
    ])
    train_loader = DataLoader(CASIADataset(train_img, train_lbl, transform), batch_size=config.BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(CASIADataset(test_img, test_lbl, transform), batch_size=config.BATCH_SIZE)
except Exception as e:
    raise RuntimeError(f"Erreur lors du chargement des données: {e}")

model = EarlyFusionSwinEffNet(
    num_classes=config.NUM_CLASSES,
    fusion_target_spatial_size=config.FUSION_TARGET_SPATIAL_SIZE,
    fusion_conv_channels=config.FUSION_CONV_CHANNELS,
    fusion_dropout=config.FUSION_DROPOUT
).to(config.DEVICE)

# Freeze des backbones
for p in model.swin.parameters(): p.requires_grad = False
for p in model.efficientnet.parameters(): p.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
scaler = GradScaler(enabled=torch.cuda.is_available())

# --- ENTRAÎNEMENT COURT ---
for epoch in range(config.EPOCHS_PHASE1):
    model.train()
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        inputs, labels = inputs.to(config.DEVICE), labels.to(config.DEVICE)
        optimizer.zero_grad()
        with autocast(device_type="cuda"):
            loss = criterion(model(inputs), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

# --- SAUVEGARDE DU MODÈLE ---
MODEL_PATH = f"best_model_{config.MODEL_NAME}.pth"
torch.save(model.state_dict(), MODEL_PATH)

# --- ANALYSE DU MODÈLE ---
model.eval()
params_total = sum(p.numel() for p in model.parameters())
macs, _ = profile(model, inputs=(torch.randn(1, 3, config.IMG_SIZE, config.IMG_SIZE).to(config.DEVICE),), verbose=False)

f1 = MulticlassF1Score(num_classes=config.NUM_CLASSES, average="macro").to(config.DEVICE)
with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc="Calcul F1"):
        preds = model(imgs.to(config.DEVICE)).argmax(dim=1)
        f1.update(preds, lbls.to(config.DEVICE))

# --- AFFICHAGE DES RÉSULTATS ---
print("\n" + "="*50)
print("RÉSULTATS FINAUX DU MODÈLE EARLY FUSION")
print("="*50)
print(f"-> Backbone 1 : Swin-Transformer (swin_tiny_patch4_window7_224)")
print(f"-> Backbone 2 : EfficientNet-B0")
print(f"-> Paramètres totaux       : {params_total / 1e6:.2f} M")
print(f"-> Complexité (GFLOPs)     : {(macs * 2) / 1e9:.2f}")
print(f"-> F1-score (macro)        : {f1.compute().item():.4f}")
print("="*50)


Montage de Google Drive...
Mounted at /content/drive


2025-06-30 15:46:39,551 - INFO - Loading pretrained weights from Hugging Face hub (timm/swin_tiny_patch4_window7_224.ms_in1k)
2025-06-30 15:46:39,788 - INFO - [timm/swin_tiny_patch4_window7_224.ms_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2025-06-30 15:46:39,805 - INFO - Missing keys (head.fc.weight, head.fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted.
2025-06-30 15:46:39,879 - INFO - Loading pretrained weights from Hugging Face hub (timm/efficientnet_b0.ra_in1k)
2025-06-30 15:46:40,113 - INFO - [timm/efficientnet_b0.ra_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2025-06-30 15:46:40,132 - WARNING - Unexpected keys (bn2.bias, bn2.num_batches_tracked, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. T


RÉSULTATS FINAUX DU MODÈLE EARLY FUSION
-> Backbone 1 : Swin-Transformer (swin_tiny_patch4_window7_224)
-> Backbone 2 : EfficientNet-B0
-> Paramètres totaux       : 31.28 M
-> Complexité (GFLOPs)     : 9.49
-> F1-score (macro)        : 0.4075
